In [ ]:
# Prathmesh Durge, 23070521109

# Slot Filling using LSTM
# Sequence Labeling for Spoken Language Understanding

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split


In [2]:
# --------------------------------------------------
# 1. Sample Dataset
# --------------------------------------------------

sentences = [
    "book a flight from delhi to mumbai",
    "book a flight from mumbai to delhi",
    "i want to fly from delhi to bangalore",
    "find a flight from chennai to delhi",
    "show flights from pune to mumbai",
    "i need a flight from kolkata to delhi",
    "book ticket from hyderabad to chennai",
    "find flights from delhi to pune",
    "i want to travel from mumbai to goa",
    "book a flight from bangalore to chennai"
]

In [3]:
# Corresponding slot labels
# O       = Outside any slot
# B-XXX   = Beginning of a slot
# I-XXX   = Inside a slot

labels = [
    "O O O O B-FROM O B-TO",
    "O O O O B-FROM O B-TO",
    "O O O B-FROM O B-TO",
    "O O O O B-FROM O B-TO",
    "O O O B-FROM O B-TO",
    "O O O O B-FROM O B-TO",
    "O O B-FROM O B-TO",
    "O O O B-FROM O B-TO",
    "O O O O B-FROM O B-TO",
    "O O O O B-FROM O B-TO"
]

labels = [x.split() for x in labels]

In [4]:

# --------------------------------------------------
# 2. Tokenize sentences
# --------------------------------------------------

tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

X = tokenizer.texts_to_sequences(sentences)

# Find maximum sentence length
max_len = max(len(x) for x in X)

X = pad_sequences(
    X,
    maxlen=max_len,
    padding="post"
)


In [5]:
# --------------------------------------------------
# 3. Create label vocabulary
# --------------------------------------------------

unique_labels = sorted(set(label for sentence in labels for label in sentence))

label_to_id = {
    label: i + 1
    for i, label in enumerate(unique_labels)
}

id_to_label = {
    i: label
    for label, i in label_to_id.items()
}

# Convert labels to numbers
y = []

for sentence_labels in labels:
    encoded = [label_to_id[label] for label in sentence_labels]

    # Padding labels with 0
    encoded += [0] * (max_len - len(encoded))

    y.append(encoded)

y = np.array(y)

In [6]:
# --------------------------------------------------
# 4. Train-Test Split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
# --------------------------------------------------
# 5. Build LSTM Model
# --------------------------------------------------

vocab_size = len(tokenizer.word_index) + 1
num_labels = len(label_to_id) + 1

model = Sequential([
    tf.keras.Input(shape=(max_len,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=64
    ),

    Bidirectional(
        LSTM(64, return_sequences=True)
    ),

    Dense(num_labels, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Display model architecture
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 8, 64)          │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 8, 128)         │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8, 4)           │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,100 (266.02 KB)

 Trainable params: 68,100 (266.02 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# --------------------------------------------------
# 6. Train the Model
# --------------------------------------------------

model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=2,
    validation_split=0.2,
    verbose=1
)


Epoch 1/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 145ms/step - accuracy: 0.5417 - loss: 1.3805 - val_accuracy: 0.5000 - val_loss: 1.3687
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.6042 - loss: 1.3582 - val_accuracy: 0.5000 - val_loss: 1.3494
Epoch 3/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5625 - loss: 1.3333 - val_accuracy: 0.5000 - val_loss: 1.3254
Epoch 4/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.5625 - loss: 1.3019 - val_accuracy: 0.5000 - val_loss: 1.2938
Epoch 5/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.5625 - loss: 1.2607 - val_accuracy: 0.5000 - val_loss: 1.2515
Epoch 6/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5625 - loss: 1.2035 - val_accuracy: 0.5000 - val_loss: 1.1935
Epoch 7/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.5625 - loss: 1.1262 - val_accuracy: 0.5000 - val_loss: 1.1161
Epoch 8/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.6042 - loss: 1.0276 - val_accuracy: 0.5625 - val_loss: 1.0258

In [15]:
# --------------------------------------------------
# 7. Evaluate Model
# --------------------------------------------------

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Accuracy:", accuracy)


Test Accuracy: 0.75


In [16]:
# --------------------------------------------------
# 8. Predict Slots
# --------------------------------------------------

def predict_slots(sentence):

    # Convert sentence to numbers
    sequence = tokenizer.texts_to_sequences([sentence])

    # Padding
    sequence = pad_sequences(
        sequence,
        maxlen=max_len,
        padding="post"
    )

    # Prediction
    prediction = model.predict(sequence, verbose=0)

    prediction = np.argmax(prediction, axis=-1)[0]

    # Get original words
    words = sentence.lower().split()

    print("\nSentence:")
    print(sentence)

    print("\nSlot Predictions:")

    for i, word in enumerate(words):
        label_id = prediction[i]

        if label_id == 0:
            label = "O"
        else:
            label = id_to_label.get(label_id, "O")

        print(f"{word:15} -> {label}")


In [17]:

# --------------------------------------------------
# 9. Test the Model
# --------------------------------------------------

predict_slots(
    "book a flight from delhi to mumbai"
)

predict_slots(
    "find a flight from pune to chennai"
)


Sentence:
book a flight from delhi to mumbai

Slot Predictions:
book            -> O
a               -> O
flight          -> O
from            -> O
delhi           -> O
to              -> O
mumbai          -> B-TO

Sentence:
find a flight from pune to chennai

Slot Predictions:
find            -> O
a               -> O
flight          -> O
from            -> O
pune            -> O
to              -> O
chennai         -> B-TO
